# Colab LLM Station 🚀

歡迎使用 **Colab LLM Station**！本 Notebook 提供在 Google Colab 上快速掛載雲端硬碟、啟動 **Tailscale SSH** 與部署 **vLLM / Ollama** 推論引擎的一鍵式流程。

---

### 📌 準備工作：設定 Tailscale Auth Key（僅需設定一次）
1. 至 [Tailscale Keys](https://login.tailscale.com/admin/settings/keys) 產生一個 Auth Key（建議勾選 **Reusable** 與 **Ephemeral**）。
2. 點擊 Colab 左側工具列的 **鑰匙圖示（Secrets / 密鑰）** 🔑。
3. 新增密鑰名稱：`TAILSCALE_AUTHKEY`，值為你的 `tskey-auth-...`。
4. 開啟 **「筆記本存取權限 (Notebook access)」**。


## 步驟 1: 掛載 Google Drive 並啟動 Tailscale SSH
執行下方儲存格後，本機即可透過 `ssh root@colab-llm-station` 或 VS Code (Remote - SSH) 直接免密碼連入！

In [ ]:
# ==============================================================================
# 獨立 SSH 啟動儲存格（基於 Tailscale SSH，免密碼、免管理金鑰）
# ==============================================================================
from google.colab import drive, userdata

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 取得剛剛存好的 Tailscale Auth Key
authkey = userdata.get('TAILSCALE_AUTHKEY')

# 3. 啟動 Tailscale 並啟用內建 SSH 伺服器
!cd /content/drive/MyDrive/colab/colab-llm-station && \
 bash setup.sh && \
 bash llm.sh tunnel tailscale up "$authkey"

print("\n" + "="*60)
print("✓ Tailscale SSH 已就緒！")
print("1. 本機終端機連線指令：")
print("   ssh root@colab-llm-station")
print("\n2. 本機 VS Code (Remote - SSH) 連線：")
print("   按 F1 -> 輸入 Remote-SSH: Connect to Host... -> 輸入 root@colab-llm-station")
print("="*60)


## 步驟 2 (選用): 啟動 LLM 推論引擎並開放 Tailnet 訪問
> 💡 **說明**：你也可以在 SSH 連進 Colab 後，直接在終端機執行推論指令；或直接在下方儲存格執行背景啟動。

In [ ]:
%%bash
cd /content/drive/MyDrive/colab/colab-llm-station

# 企業級 GPU (A100 / H100): 啟動 vLLM (Port 8000)
bash llm.sh tunnel tailscale serve 8000
bash llm.sh vllm serve Qwen/Qwen2.5-Coder-32B-Instruct

# 若為標準級 GPU (T4 / L4)，請取消註解改跑 Ollama (Port 11434):
# bash llm.sh tunnel tailscale serve 11434
# bash llm.sh ollama serve


## 步驟 3: 檢視系統與網路狀態

In [ ]:
!cd /content/drive/MyDrive/colab/colab-llm-station && bash llm.sh status
